
### import needed packages and read the combined dataset

In [1]:
import pandas as pd

In [2]:
# Define dataset paths
dataset_files = [
    'CEAS_08.csv',
    'Nazario.csv',
    'Nazario_2.csv',
    'Nazario_5.csv',
    'Nigerian_5.csv',
    'Nigerian_Fraud.csv',
    'SpamAssasin.csv',
    'TREC_07.csv'
]

# Load _datasets into dictionary
datasets = {}

for file in dataset_files:
    try:
        name = file.replace('.csv', '')
        datasets[name] = pd.read_csv(f'datasets/raw/{file}')
        print(f"✓ Loaded {file}")
    except Exception as e:
        print(f"✗ Error loading {file}: {e}")

print(f"\n✓ Successfully loaded {len(datasets)} _datasets")

✓ Loaded CEAS_08.csv
✓ Loaded Nazario.csv
✓ Loaded Nazario_2.csv
✓ Loaded Nazario_5.csv
✓ Loaded Nigerian_5.csv
✓ Loaded Nigerian_Fraud.csv
✓ Loaded SpamAssasin.csv
✓ Loaded TREC_07.csv

✓ Successfully loaded 8 datasets


### Check all dataset dimensions columns etc, so we know how to properly combine them

In [3]:
dimensions_df = pd.DataFrame({
    'Dataset': list(datasets.keys()),
    'Rows': [df.shape[0] for df in datasets.values()],
    'Columns': [df.shape[1] for df in datasets.values()],
    'Memory (MB)': [df.memory_usage(deep=True).sum() / 1024**2 for df in datasets.values()]
})

display(dimensions_df)

print(f"\nTotal Rows: {dimensions_df['Rows'].sum():,}")
print(f"Total Columns: {dimensions_df['Columns'].sum()}")
print(f"Total Memory: {dimensions_df['Memory (MB)'].sum():.2f} MB")

,Dataset,Rows,Columns,Memory (MB)
0,CEAS_08,39154,7,80.879686
1,Nazario,1565,7,12.584010
2,Nazario_2,1565,7,12.584010
3,Nazario_5,3065,7,17.079152
4,Nigerian_5,6331,7,20.654429
5,Nigerian_Fraud,3332,7,10.044590
6,SpamAssasin,5809,7,16.078164
7,TREC_07,53757,7,123.639003



Total Rows: 114,578
Total Columns: 56
Total Memory: 293.54 MB


In [4]:
for name, df in datasets.items():
    print(f"\n{'='*70}")
    print(f"DATASET: {name}")
    print(f"{'='*70}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

    print("Columns:")
    for i, col in enumerate(df.columns, 1):
        dtype = df[col].dtype
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100
        print(f"  {i:2d}. {col:30s} | {str(dtype):10s} | Nulls: {null_count:5d} ({null_pct:5.2f}%)")

    print(f"\n{'─'*70}")


DATASET: CEAS_08
Shape: 39,154 rows × 7 columns

Columns:
   1. sender                         | str        | Nulls:     0 ( 0.00%)
   2. receiver                       | str        | Nulls:   462 ( 1.18%)
   3. date                           | str        | Nulls:     0 ( 0.00%)
   4. subject                        | str        | Nulls:    28 ( 0.07%)
   5. body                           | str        | Nulls:     0 ( 0.00%)
   6. label                          | int64      | Nulls:     0 ( 0.00%)
   7. urls                           | int64      | Nulls:     0 ( 0.00%)

──────────────────────────────────────────────────────────────────────

DATASET: Nazario
Shape: 1,565 rows × 7 columns

Columns:
   1. sender                         | str        | Nulls:     0 ( 0.00%)
   2. receiver                       | str        | Nulls:    96 ( 6.13%)
   3. date                           | str        | Nulls:     1 ( 0.06%)
   4. subject                        | str        | Nulls:     4 ( 0.26

In [5]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df['label'].value_counts())
    print(f"{'─'*40}")


CEAS_08
label
1    21842
0    17312
Name: count, dtype: int64
────────────────────────────────────────

Nazario
label
1    1565
Name: count, dtype: int64
────────────────────────────────────────

Nazario_2
label
1    1565
Name: count, dtype: int64
────────────────────────────────────────

Nazario_5
label
1    1565
0    1500
Name: count, dtype: int64
────────────────────────────────────────

Nigerian_5
label
1    3332
0    2999
Name: count, dtype: int64
────────────────────────────────────────

Nigerian_Fraud
label
1    3332
Name: count, dtype: int64
────────────────────────────────────────

SpamAssasin
label
0    4091
1    1718
Name: count, dtype: int64
────────────────────────────────────────

TREC_07
label
1    29399
0    24358
Name: count, dtype: int64
────────────────────────────────────────


### Combine the datasets into a single dataset and load the knew unified dataset

In [6]:
import os

# Add source column to track which dataset each row came from
for name, df in datasets.items():
    df['source_dataset'] = name
    print(f"✓ Added source column to {name}")

# Combine all _datasets
combined_df = pd.concat(datasets.values(), ignore_index=True)

print(f"\n{'='*60}")
print(f"Combined Dataset Shape: {combined_df.shape[0]:,} rows × {combined_df.shape[1]} columns")
print(f"{'='*60}\n")

# Display value counts by source
print("Records per source dataset:")
display(combined_df['source_dataset'].value_counts().sort_index())

# Export to CSV - create directory if it doesn't exist
output_file = 'datasets/processed/combined_dataset.csv'
os.makedirs(os.path.dirname(output_file), exist_ok=True)
combined_df.to_csv(output_file, index=False)
print(f"\n✓ Combined dataset exported to: {output_file}")

✓ Added source column to CEAS_08
✓ Added source column to Nazario
✓ Added source column to Nazario_2
✓ Added source column to Nazario_5
✓ Added source column to Nigerian_5
✓ Added source column to Nigerian_Fraud
✓ Added source column to SpamAssasin
✓ Added source column to TREC_07

Combined Dataset Shape: 114,578 rows × 8 columns

Records per source dataset:


source_dataset
CEAS_08           39154
Nazario            1565
Nazario_2          1565
Nazario_5          3065
Nigerian_5         6331
Nigerian_Fraud     3332
SpamAssasin        5809
TREC_07           53757
Name: count, dtype: int64


✓ Combined dataset exported to: datasets/processed/combined_dataset.csv


In [7]:
# Load the unified dataset
unified_df = pd.read_csv('datasets/processed/combined_dataset.csv', low_memory=False)
unified_df

,sender,receiver,date,subject,body,label,urls,source_dataset
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1,CEAS_08
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1,CEAS_08
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1,CEAS_08
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1,CEAS_08
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1,CEAS_08
...,...,...,...,...,...,...,...,...
114573,SCC <Gerry.Rossi4360@kinki-kids.com>,Deficient <deficient@flax9.uwaterloo.ca>,"Fri, 06 Jul 2007 06:53:36 -0400",Job: just for you.,\n\n\n\nWhile we may have high ...,1,1,TREC_07
114574,Sydney Car Centre <Merrill8783@168city.com>,Gnitpick <gnitpick@flax9.uwaterloo.ca>,"Fri, 06 Jul 2007 06:59:51 -0400",the reply for your request for a job place [le...,\n\n\n\nWhile we may have high ...,1,1,TREC_07
114575,Philippe Grosjean <phgrosjean@sciviews.org>,Duncan Murdoch <murdoch@stats.uwo.ca>,"Fri, 06 Jul 2007 12:57:17 +0200","Re: [R] Me again, about the horrible documenta...","For those who are interested, I just cook a li...",0,1,TREC_07
114576,Bernhard Wellhöfer <Bernhard.Wellhoefer@gaia-g...,r-help@stat.math.ethz.ch,"Fri, 06 Jul 2007 12:43:12 +0200",Re: [R] RODBC problem,"Hello,\n\nas I wrote I call\n\n sqlFetch(chan...",0,1,TREC_07
